# 💸 Módulo 07 - Notebook 02: Deflactación IPC y CAGR

## 📉 Ajuste por Inflación y Tasa de Crecimiento Anual Compuesta

**Libro:** Saliendo de lo Pandito  
**Módulo:** 07 - Series de Tiempo Financieras  
**Duración estimada:** 65 minutos  
**Dificultad:** 🟠 Intermedio-Avanzado  
**Plataforma:** Databricks Free Edition

---

## 🎯 Objetivos de aprendizaje

Al finalizar este notebook serás capaz de:

✅ **Deflactar** series financieras usando IPC  
✅ **Calcular** valores reales (moneda constante)  
✅ **Aplicar** CAGR (Tasa de Crecimiento Anual Compuesta)  
✅ **Comparar** crecimiento nominal vs real  
✅ **Analizar** el impacto de la inflación

---

## 📋 Pre-requisitos

* ✅ Notebook 07_01 completado (Resampling y Rolling)
* ✅ Conocimiento de series de tiempo
* ✅ Familiaridad con conceptos de inflación

---

## 📚 Contenido

1. Inflación y el Índice de Precios al Consumidor (IPC)
2. Deflactación: Valores Nominales vs Reales
3. Cálculo de Valores Reales
4. CAGR: Tasa de Crecimiento Anual Compuesta
5. Comparación: Crecimiento Nominal vs Real
6. Caso Integrador: Análisis de Ventas Ajustadas

---

## 💡 Por qué importa

**Sin ajustar por inflación, no conoces el crecimiento real:**

* 📈 **Ventas +50% en 3 años** → ¿Crecimiento real o solo inflación?
* 💰 **Salario +20% anual** → ¿Poder adquisitivo mayor o menor?
* 🏪 **Ingresos constantes** → Con inflación = **pérdida real**

**Deflactar = Ver la realidad detrás de los números**

In [0]:
import pandas as pd
import numpy as np

print("💾 CARGANDO DATOS REALES DESDE UNITY CATALOG")
print("="*70)

CATALOG = "pandito_ds"
SCHEMA = "default"

try:
    # Cargar tabla de ventas de Los Andes Market
    df_raw = spark.table(f"{CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3").toPandas()
    df_raw['fecha'] = pd.to_datetime(df_raw['fecha'])
    
    # Crear serie temporal agregada
    df_ts = df_raw.groupby('fecha')['ventas'].sum().sort_index().to_frame()
    df_ts.columns = ['ventas_nominales']
    
    # Simular IPC (aproximación de inflación Argentina)
    # Base 100 en el primer mes, inflación mensual promedio ~3%
    n_meses = len(df_ts)
    inflacion_mensual = 0.03  # 3% mensual (ajustar según contexto real)
    df_ts['ipc'] = 100 * ((1 + inflacion_mensual) ** np.arange(n_meses))
    
    # Calcular ventas reales (deflactadas)
    df_ts['ventas_reales'] = (df_ts['ventas_nominales'] / df_ts['ipc']) * 100
    
    # Calcular crecimiento
    df_ts['crecimiento_nominal_pct'] = df_ts['ventas_nominales'].pct_change() * 100
    df_ts['crecimiento_real_pct'] = df_ts['ventas_reales'].pct_change() * 100
    
    print(f"\n✅ Datos reales cargados exitosamente")
    print(f"   📅 Período: {df_ts.index.min().strftime('%Y-%m-%d')} a {df_ts.index.max().strftime('%Y-%m-%d')}")
    print(f"   📊 Meses totales: {len(df_ts)}")
    print(f"   📍 Ubicación: Mendoza, Argentina")
    
    print(f"\n📋 Columnas disponibles en df_ts:")
    print(f"   • ventas_nominales: Ventas en pesos corrientes")
    print(f"   • ipc: Índice de Precios al Consumidor (base 100)")
    print(f"   • ventas_reales: Ventas deflactadas (moneda constante)")
    print(f"   • crecimiento_nominal_pct: % crecimiento nominal")
    print(f"   • crecimiento_real_pct: % crecimiento real")
    
    print(f"\n🎯 Este notebook usará datos REALES para deflactación")
    
    USAR_DATOS_REALES = True
    
except Exception as e:
    print(f"\n⚠️  No se pudo cargar la tabla de Unity Catalog")
    print(f"   Error: {e}")
    print(f"\n📝 Solución:")
    print(f"   1. Ejecuta primero: 00_05_Preparacion_Datos_Empresariales.ipynb")
    print(f"   2. Verifica que la tabla exista: {CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    print(f"\n   Continuando con datos sintéticos...")
    
    df_ts = None
    USAR_DATOS_REALES = False

print("\n" + "="*70)

## 📚 Deflactación y CAGR

### 📉 Inflación y el IPC

**Inflación:** Aumento generalizado y sostenido de los precios.

**IPC (Índice de Precios al Consumidor):** Mide la evolución del costo de una canasta de bienes y servicios.

**Ejemplo:**
```
Año    IPC    Interpretación
2020   100    Base
2021   115    Los precios subieron 15%
2022   132    Los precios subieron 32% desde 2020
```

---

### 💸 Valores Nominales vs Reales

#### **Valor Nominal**
* **Definición:** Valor en **pesos corrientes** (de cada año)
* **Problema:** No considera la pérdida de poder adquisitivo

#### **Valor Real**
* **Definición:** Valor ajustado por inflación (**moneda constante**)
* **Ventaja:** Muestra el crecimiento/decrecimiento verdadero

---

### 🧮 Fórmula de Deflactación

```
Valor Real = (Valor Nominal / IPC) × 100
```

**Ejemplo:**
```
Año    Ventas Nominales    IPC    Ventas Reales
2020   $100,000            100    $100,000
2021   $120,000            115    $104,348  (120,000 / 115 × 100)
2022   $150,000            132    $113,636  (150,000 / 132 × 100)
```

**Interpretación:**
* Crecimiento **nominal** 2020→2022: +50%
* Crecimiento **real** 2020→2022: +13.6%
* La inflación "se comió" gran parte del crecimiento

---

### 📈 CAGR (Compound Annual Growth Rate)

**Definición:** Tasa de crecimiento anual compuesta.

**Fórmula:**
```
CAGR = (Valor_Final / Valor_Inicial)^(1/n_años) - 1
```

**En Python:**
```python
def calcular_cagr(valor_inicial, valor_final, n_anos):
    return (valor_final / valor_inicial) ** (1 / n_anos) - 1

cagr = calcular_cagr(100000, 150000, 3)
print(f"CAGR: {cagr*100:.2f}%")  # 14.47%
```

---

### 🔢 CAGR Nominal vs Real

**Ejemplo completo:**
```
Ventas 2020: $100,000  (IPC 100)
Ventas 2023: $180,000  (IPC 150)

CAGR Nominal = (180,000 / 100,000)^(1/3) - 1 = 21.6%

Ventas Reales 2020: $100,000
Ventas Reales 2023: $120,000  (180,000 / 150 × 100)

CAGR Real = (120,000 / 100,000)^(1/3) - 1 = 6.3%
```

**Interpretación:**
* 📈 Crecimiento **nominal**: 21.6% anual
* 💰 Crecimiento **real**: 6.3% anual
* 🔥 Inflación **implicó**: ~15% anual

---

### 🎯 ¿Cuándo usar cada métrica?

| Métrica | Uso |
|---------|-----|
| **Valores Nominales** | Reportes contables, estados financieros |
| **Valores Reales** | Análisis de tendencias de largo plazo |
| **CAGR Nominal** | Crecimiento aparente (incluye inflación) |
| **CAGR Real** | Crecimiento verdadero (sin inflación) |

---

### 💡 Regla de Oro

👉 **Para series de más de 1 año, SIEMPRE deflactar**  
👉 **Comparar manzanas con manzanas = usar moneda constante**

In [0]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("💸 DEFLACTACIÓN IPC Y CAGR")
print("="*70)

print(f"\nVersión de Pandas: {pd.__version__}")
print(f"Versión de NumPy: {np.__version__}")

print("\n🎯 En este notebook aprenderás:")
print("  • Deflactación: Ajustar series por inflación (IPC)")
print("  • Valores nominales vs reales (moneda constante)")
print("  • CAGR: Tasa de Crecimiento Anual Compuesta")
print("  • Comparación: Crecimiento nominal vs real")

print("\n📖 Fórmulas clave:")
print("  - Valor Real = (Valor Nominal / IPC) × 100")
print("  - CAGR = (Valor_Final / Valor_Inicial)^(1/n) - 1")

print("\n" + "="*70)
print("✅ Librerías cargadas correctamente")

## 🎓 Conclusiones del notebook 07_02

### ✅ Lo que aprendiste

1. **Inflación y IPC:**
   - El IPC mide la evolución del costo de una canasta de bienes y servicios
   - Base 100 en un período de referencia; valores > 100 indican inflación acumulada
   - Sin ajustar por inflación, el crecimiento aparente es engañoso

2. **Deflactación (Nominal → Real):**
   - `Valor Real = (Valor Nominal / IPC) × 100`
   - Convierte pesos corrientes a moneda constante
   - Permite comparar valores de diferentes años en términos reales

3. **CAGR (Tasa de Crecimiento Anual Compuesta):**
   - `CAGR = (Valor_Final / Valor_Inicial)^(1/n) - 1`
   - Mide el crecimiento promedio anual suavizado
   - Útil para comparar trayectorias de diferentes series

4. **CAGR Nominal vs Real:**
   - Nominal incluye inflación → sobreestima el crecimiento verdadero
   - Real elimina inflación → muestra el poder adquisitivo real
   - La diferencia entre ambos = inflación promedio anual implícita

5. **Caso integrador:**
   - Ventas nominales pueden crecer +50% pero reales solo +13%
   - Reportes contables usan nominal; análisis estratégico usa real
   - Deflactar series de más de 1 año es obligatorio para decisiones

---

### 🎯 Reglas de Oro

👉 **Regla #1: Para series de más de 1 año, SIEMPRE deflactar**
```python
# MALO: comparar ventas nominales de 2020 vs 2023
ventas_2020 = 100000
ventas_2023 = 150000
crecimiento = (ventas_2023 / ventas_2020 - 1) * 100  # +50% — engañoso

# BUENO: deflactar antes de comparar
ventas_reales_2020 = ventas_2020 / ipc_2020 * 100
ventas_reales_2023 = ventas_2023 / ipc_2023 * 100
crecimiento_real = (ventas_reales_2023 / ventas_reales_2020 - 1) * 100
```

👉 **Regla #2: CAGR con valores REALES, no nominales**
```python
# MALO: CAGR nominal (infla el crecimiento)
cagr = (valor_final_nom / valor_ini_nom) ** (1/n) - 1

# BUENO: CAGR real (crecimiento verdadero)
cagr = (valor_final_real / valor_ini_real) ** (1/n) - 1
```

👉 **Regla #3: Documentar la fuente del IPC**
```python
# MALO: usar un número arbitrario
ventas_reales = ventas / 150 * 100  # ¿De dónde sale 150?

# BUENO: documentar fuente y base
# IPC base 100 = diciembre 2019 (INDEC)
# Fuente: https://www.indec.gob.ar
ventas_reales = (ventas / df['ipc']) * 100
```

---

### 📊 Guía de Decisión

| Situación | Métrica |
|-----------|--------|
| Reportes contables / estados financieros | Valores nominales |
| Análisis de tendencia de largo plazo | Valores reales (deflactados) |
| Crecimiento aparente (incluye inflación) | CAGR nominal |
| Crecimiento verdadero (sin inflación) | CAGR real |
| Comparar años diferentes | Moneda constante (deflactar) |
| Comparar sucursales mismo período | Valores nominales (no requiere IPC) |
| Evaluar poder adquisitivo | Valores reales |
| Proyectar ventas futuras | Usar CAGR real + inflación esperada |

---

<div style="background: linear-gradient(90deg, #2563eb 0%, #60a5fa 100%); padding: 20px; border-radius: 10px; color: white; text-align: center;">
  <h3>💸 ¡Deflactación y CAGR dominados!</h3>
  <p><i>"Deflactar es ver la realidad detrás de los números: el crecimiento real, no la ilusión de la inflación."</i></p>
</div>